### Tool

Model can request to call tools that perform a task such as calculating, running code, searching on web, or fetching data.
Tools are paring of
 1. A Schema, including name of tool, description or argument definition, and parameters.(JSON Schema)
 2. A function that implements the tool.

In [9]:
import os
from langchain_groq import ChatGroq


model=ChatGroq(model="openai/gpt-oss-20b")
response=model.invoke("What is weather like in Pune?")
response.content

'I’m not able to fetch real‑time weather data, but I can give you a general idea of what Pune’s climate is like throughout the year:\n\n| Season | Typical Weather | Temperature Range (°C) | Key Features |\n|--------|-----------------|------------------------|--------------|\n| **Winter (Dec – Feb)** | Cool, dry, pleasant | 10\u202f–\u202f30\u202f°C | Nights can be chilly (10–12\u202f°C). Daytime is usually sunny. |\n| **Spring (Mar – Apr)** | Warm, dry, occasional dust | 15\u202f–\u202f35\u202f°C | Warm days, nights still mild. |\n| **Summer (May – June)** | Hot, humid, dry | 20\u202f–\u202f38\u202f°C | Heat peaks in late May. Humidity starts rising toward June. |\n| **Monsoon (July – September)** | Rainy, humid | 20\u202f–\u202f35\u202f°C | Heavy showers, especially in July & August. Humidity can be >80\u202f%. |\n| **Post‑monsoon (Oct – Nov)** | Mild, pleasant | 15\u202f–\u202f30\u202f°C | Clear skies, lower humidity. |\n\n### Quick Tips for Checking Current Conditions\n- **Weather A

In [13]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at location"""
    return f"It's a sunny {location}"

model_with_tool=model.bind_tools([get_weather])

In [14]:
response=model_with_tool.invoke("What is weather like in Baramati or in Pune")
print(response)
for tool_call in response.tool_calls:
    # view tool call made by model
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather function with location either "Baramati" or "Pune". Probably choose one. The question: "What is weather like in Baramati or in Pune". Could ask for both. We\'ll provide function calls for both.', 'tool_calls': [{'id': 'fc_16e7afc5-9f38-4739-a1c8-6703b1a7f042', 'function': {'arguments': '{"location":"Baramati"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 130, 'total_tokens': 208, 'completion_time': 0.081921321, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.007339936, 'prompt_tokens_details': None, 'queue_time': 0.282509835, 'total_time': 0.089261257}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66891002f6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a03325-c724-7c12-9820-de8f72498865-0' tool_calls=[{'name': 'get

In [15]:
#step 1:Model generate tool call
message=[{"role":"user","content":"Whats's the weather in Pune?"}]
ai_msg=model_with_tool.invoke(message)
message.append(ai_msg)

#step 2:Execute tool and collect result
for tool_call in ai_msg.tool_calls:
   #Execute the tool with generated argument
   tool_result=get_weather.invoke(tool_call)
   message.append(tool_result)
    
#step 3:pass result back to model fro final result
final_response=model_with_tool.invoke(message)
print(final_response.text)

It’s a sunny Pune.


In [16]:
message

[{'role': 'user', 'content': "Whats's the weather in Pune?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the function get_weather with location "Pune".', 'tool_calls': [{'id': 'fc_058f5cf2-e50c-4744-9275-b00a78b1ab5a', 'function': {'arguments': '{"location":"Pune"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 126, 'total_tokens': 165, 'completion_time': 0.054249473, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.00600616, 'prompt_tokens_details': None, 'queue_time': 0.368440536, 'total_time': 0.060255633}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5979a0e1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a03325-c8e2-7b32-8346-73d7b907c253-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': 'fc_058f5cf2-e50c-4744-9275-b00a78b1ab5a'